# Day 1 | ILT 1: Problem Statement & 4-Source Architecture (GlobalMart)
### GlobalMart Data Engineering Bootcamp
---
**Session Time:** 9:30 AM - 10:30 AM  
**Goal:** Understand WHO GlobalMart is, WHAT problem they have, and WHAT we will build over 3 weeks.

---
**INSTRUCTOR NOTE:**  
This is the FIRST session of the entire bootcamp. No code yet - this session is purely about setting context.  
Students need to understand the business problem before they can understand any technical solution.  

By the end of this session, every student should be able to answer:  
- *What does GlobalMart do?*  
- *What is their data problem?*  
- *What are the 4 sources of data?*  
- *What will we build over 3 weeks?*

The last 10 minutes have a live code demo where we peek at the actual dataset.

## Learning Objectives

By the end of this session, students will be able to:

1. Describe what GlobalMart is and its data challenges
2. Identify the 4 source systems and what data each produces
3. Explain why GlobalMart needs a Lakehouse platform
4. Map the 10 dataset tables to their source systems
5. Describe the 3-week journey at a high level

---
## Section 1: Who is GlobalMart?

**INSTRUCTOR NOTE:**  
Start with this. Tell students: *'For the next 3 weeks, we are Data Engineers at a company called GlobalMart. Everything we build is for this company.'*

---

**GlobalMart** is a mid-sized e-commerce company - think of it as a smaller version of Amazon or Flipkart.

### What does GlobalMart do?

- Sells products across multiple categories (electronics, clothing, home goods, etc.)
- Has customers placing orders online
- Works with multiple suppliers to source products
- Offers multiple payment options (credit card, UPI, cash on delivery)
- Has multiple shipping tiers (standard, express, same-day)
- Handles product returns and refunds

### Key Numbers (GlobalMart scale):

| Metric | Value |
|--------|-------|
| Customers | ~10,000+ |
| Products in catalog | ~100,000+ (products.csv is 100 MB!) |
| Orders per day | Thousands |
| Data sources | 4 different systems |
| Tables we work with | 10 |

**INSTRUCTOR NOTE:**  
Mention: *'The data you have in your ADLS is real-scale data. products.csv alone is 100 MB. This is why we use Spark and not Excel.'

---
## Section 2: The Problem Statement

> **Instructor Note:**  
> This section explains **why** the project exists. Read it with energy—the entire bootcamp is focused on solving these business challenges.

---

## GlobalMart's Current Situation

GlobalMart's data is spread across **four different systems** that do not communicate with each other.

| Source System | Data Stored | Data Format |
|--------------|------------|-------------|
| **Supabase PostgreSQL** | Orders, Customers, Products, Suppliers, Payments, Returns | Structured SQL |
| **Public APIs** | Exchange Rates, Shipping Zones, Enrichment Data | JSON |
| **Neo4j Graph Database** | Customer-Product Relationships, Recommendations | Graph Data |
| **Unstructured Files** | Product Images, Supplier Contracts, Reviews | JPG, PDF, TXT |

---

## Business Questions That Are Difficult to Answer

### Question 1
**Which products have the highest return rate?**

**Current Challenge**
- Return information exists in PostgreSQL
- Product relationships exist in Neo4j
- The systems cannot be joined easily

---

### Question 2
**What is our revenue in USD versus INR today?**

**Current Challenge**
- Order amounts are stored in PostgreSQL
- Exchange rates come from an external API
- No automatic integration exists

---

### Question 3
**Which customers are most likely to buy together?**

**Current Challenge**
- Order history exists in PostgreSQL
- Relationship data exists in Neo4j
- Analysts must manually combine data

---

### Question 4
**Do supplier contracts match what we are paying?**

**Current Challenge**
- Contracts are stored as PDF files
- Payment information is stored in PostgreSQL
- No centralized way to compare them

---

### Question 5
**Show last month's revenue by product category.**

**Current Challenge**
- Requires joining multiple tables
- Analysts spend hours creating reports
- Business users must wait for answers

---

## GlobalMart's Three Major Pain Points

### 1. Data Silos
Data is spread across multiple systems and cannot be analyzed together efficiently.

### 2. No Single Source of Truth
Different departments use different data sources and often report different numbers.

### 3. Slow Decision Making
Business teams wait hours or even days for reports that should be available immediately.

---

## The Solution

GlobalMart needs a **Modern Azure Lakehouse Platform** that can:

- Ingest data from PostgreSQL, APIs, Neo4j, and file storage automatically
- Store all raw data in **Azure Data Lake Storage Gen2 (Bronze Layer)**
- Clean and standardize data using **Delta Lake (Silver Layer)**
- Create business-ready analytical models using **Gold Layer tables**
- Run pipelines automatically through **Databricks Workflows**
- Provide governance, security, lineage, and access control using **Unity Catalog**

---

## Target Architecture

```text
                    +----------------------+
                    |    Source Systems    |
                    +----------------------+

     PostgreSQL      APIs       Neo4j       Files
          |            |           |           |
          +------------+-----------+-----------+
                               |
                               v

                    +----------------------+
                    | Bronze Layer (Raw)   |
                    | ADLS Gen2            |
                    +----------------------+
                               |
                               v

                    +----------------------+
                    | Silver Layer         |
                    | Clean Delta Tables   |
                    +----------------------+
                               |
                               v

                    +----------------------+
                    | Gold Layer           |
                    | Business Models      |
                    +----------------------+
                               |
                               v

                    Dashboards & Analytics
```

---

## End Goal

Instead of multiple departments getting different answers from different systems:

**One Gold Table → One Revenue Number → One Version of the Truth**

This is the Lakehouse platform we will build throughout the bootcamp.

> **Instructor Question:**  
> Have you ever seen Finance report one revenue number while Sales reports a different revenue number?
>
> That happens because of **Data Silos**.
>
> The Lakehouse architecture we build ensures everyone uses the same trusted source of data.

---
## Section 3: The Four Data Sources

> **Instructor Note:**  
> Spend approximately 15 minutes on this section.
>
> Each source requires a different ingestion approach in Databricks.
>
> **Important:** The primary focus of this bootcamp is **Source 1 (Supabase PostgreSQL)**. Sources 2, 3, and 4 are introduced through demonstrations to expose students to real-world ingestion patterns.

---

# Source 1: Supabase (PostgreSQL) — Primary Source

## What Is It?

Supabase is GlobalMart's primary transactional database.

Every customer registration, order placement, payment, product update, and return transaction is stored here.

Supabase is built on top of **PostgreSQL**, one of the most widely used relational databases in the world.

### Source Details

| Property | Description |
|-----------|-------------|
| Technology | Supabase (PostgreSQL) |
| Data Format | Structured tables (rows and columns) |
| Data Frequency | Continuously updated throughout the day |
| Connection Method | JDBC and CDC (Change Data Capture) |
| CDC Source | PostgreSQL Write-Ahead Log (WAL) |
| Main Tables | customers, orders, order_items, payments, addresses, returns, products, suppliers |
| Main Challenge | Running analytics directly on production systems can impact application performance |
| Bootcamp Coverage | Covered throughout all three weeks |

### Understanding WAL

**WAL (Write-Ahead Log)** is a PostgreSQL feature that records every database change before it is committed.

This allows us to:

- Capture inserts
- Capture updates
- Capture deletes
- Replicate changes in near real time

Instead of repeatedly reading the entire database, we can simply process new changes from the WAL.

### Why This Source Matters

This is where most business transactions occur.

Most of the Bronze, Silver, and Gold layer pipelines in this bootcamp are built using data originating from PostgreSQL.

> **Instructor Note:**  
> Explain that the CSV files used in the bootcamp represent exports from PostgreSQL tables.
>
> In a real production environment, Databricks would connect directly to PostgreSQL through JDBC rather than reading CSV files.

---

# Source 2: Public APIs (REST)

## What Is It?

Not all business data is stored internally.

Organizations often enrich their data using external services that provide information through APIs.

GlobalMart uses public APIs for information such as:

- Currency exchange rates
- Shipping zones
- Geographic information
- Country and region details

### Source Details

| Property | Description |
|-----------|-------------|
| Technology | REST API over HTTP |
| Data Format | JSON |
| Examples | Exchange rates, shipping zones, country information |
| Connection Method | Python requests library |
| Main Challenges | Pagination, rate limits, schema changes |
| Bootcamp Coverage | Demonstrated once |
| Free Tool Used | frankfurter.app |

### Typical Workflow

1. Send an HTTP request
2. Receive JSON response
3. Store raw JSON in Bronze
4. Transform JSON into structured tables

### Example Use Case

To calculate revenue in multiple currencies:

- Orders come from PostgreSQL
- Exchange rates come from an API

Both datasets must be combined inside the Lakehouse.

### Free API Used

```text
https://api.frankfurter.app/latest
```

Returns current foreign exchange rates in JSON format.

No registration or API key is required.

---

# Source 3: Neo4j Graph Database

## What Is It?

Neo4j is a graph database.

Unlike relational databases that store rows and tables, graph databases store:

- Nodes (entities)
- Relationships (connections between entities)

### Example

```text
(Customer)
     |
   BOUGHT
     |
(Product)
     |
 SUPPLIED_BY
     |
(Supplier)
```

Graph databases are excellent for relationship-based analysis and recommendation systems.

### Source Details

| Property | Description |
|-----------|-------------|
| Technology | Neo4j Graph Database |
| Query Language | Cypher |
| Data Stored | Customer, Product, Supplier relationships |
| Connection Method | Cypher query → Tabular data → Bronze |
| Revisited Later | GraphFrames analytics on Day 15 |
| Bootcamp Coverage | Demo in Week 1 and GraphFrames exercise in Week 3 |
| Free Tool Used | Neo4j AuraDB Free |

### Example Cypher Query

```cypher
MATCH (c:Customer)-[:BOUGHT]->(p:Product)
RETURN c, p
```

### Why Use Graph Databases?

Questions involving relationships are often easier to solve using graphs.

Examples:

- Which customers buy similar products?
- Which suppliers are connected to high-return products?
- Which products are frequently purchased together?

### Free Tool Used

```text
neo4j.com/cloud/aura
```

Provides a free cloud-hosted Neo4j environment.

---

# Source 4: Unstructured Files

## What Is It?

Not all business data is stored in databases.

Many organizations also store:

- Images
- Documents
- Contracts
- Text files

These files are considered unstructured data.

### Examples

- Product images (JPG, PNG)
- Supplier contracts (PDF)
- Customer reviews (TXT)

### Source Details

| Property | Description |
|-----------|-------------|
| Technology | Files stored in ADLS |
| Data Format | JPG, PNG, PDF, TXT |
| Connection Method | Databricks Autoloader |
| Stored Information | Binary file plus metadata |
| Extracted Information | File name, size, path, timestamps |
| Bootcamp Coverage | Demonstrated once |
| Free Tool Used | Sample files uploaded to ADLS |

### Typical Workflow

```text
Files Uploaded
       ↓
Autoloader Detects New Files
       ↓
Bronze Layer Stores Files
       ↓
Metadata Extracted
```

### Important Note

In this bootcamp we only extract file metadata.

We do **not** perform:

- OCR
- Image classification
- Machine learning
- Document intelligence

---

# Comparing the Four Sources

| Feature | Supabase (PostgreSQL) | Public API | Neo4j | Unstructured Files |
|----------|----------------------|------------|--------|-------------------|
| Data Format | SQL Tables | JSON | Graph Data | Binary Files |
| Connection Method | JDBC + CDC | REST API | Cypher Queries | Autoloader |
| Main Challenge | Incremental Loading | Rate Limits | Graph Flattening | Metadata Extraction |
| Bootcamp Coverage | Extensive | Demo | Demo + Day 15 | Demo |
| Free Tool | Supabase | frankfurter.app | Neo4j AuraDB Free | Sample Files |

---

# Key Takeaway

The modern data engineer rarely works with only one type of data source.

A typical enterprise environment includes:

- Relational databases
- APIs
- Graph databases
- Unstructured files

The goal of the Lakehouse is to bring all of these sources into a single platform for analytics.

> **Instructor Note:**  
> Reinforce that approximately **80% of the bootcamp focuses on PostgreSQL, ADLS, Delta Lake, and Databricks.**
>
> The remaining sources are included to expose students to real-world ingestion scenarios they will encounter in production environments.

---
## Section 4: Our Dataset – Tables Mapped to Sources

> **Instructor Note:**  
> This section connects the architecture and data sources to the actual files students will use.
>
> Open the ADLS container and show the files inside the `raw/` folder.
>
> Explain that these CSV files are exports from PostgreSQL and represent the data that would normally be ingested directly through JDBC.

---

# Where Do Our 10 CSV Files Come From?

All 10 CSV files originate from **Source 1: Supabase PostgreSQL**.

In a production environment:

```text
PostgreSQL
     ↓
JDBC / CDC
     ↓
ADLS Bronze Layer
```

For this bootcamp, we use exported CSV files instead of connecting directly to the database.

---

# Dataset Overview

| File | Source System | Description |
|--------|--------------|-------------|
| customers.csv | Supabase PostgreSQL | Customer profile information |
| orders.csv | Supabase PostgreSQL | Orders placed by customers |
| order_items.csv | Supabase PostgreSQL | Individual products within each order |
| payments.csv | Supabase PostgreSQL | Payment transactions |
| addresses.csv | Supabase PostgreSQL | Customer delivery addresses |
| returns.csv | Supabase PostgreSQL | Returned orders and refund details |
| products.csv | Supabase PostgreSQL | Product catalog |
| suppliers.csv | Supabase PostgreSQL | Supplier information |
| payment_methods.csv | Supabase PostgreSQL | Payment method lookup table |
| shipping_tier.csv | Supabase PostgreSQL | Shipping service lookup table |

---

# Understanding Each Table

## customers.csv

Stores customer profile information.

### Example Data

| Field |
|---------|
| Customer ID |
| Name |
| Email |
| City |
| Signup Date |

### Business Questions

- How many customers do we have?
- Which cities generate the most orders?
- How many new customers joined last month?

---

## orders.csv

Stores every order placed by customers.

### Example Data

| Field |
|---------|
| Order ID |
| Customer ID |
| Order Date |
| Order Status |
| Order Amount |

### Business Questions

- Total revenue
- Monthly sales trends
- Order volume analysis

---

## order_items.csv

Stores the products purchased within each order.

### Example Data

| Field |
|---------|
| Order ID |
| Product ID |
| Quantity |
| Unit Price |

### Business Questions

- Best-selling products
- Product-level revenue
- Category performance

---

## payments.csv

Stores payment information for each order.

### Example Data

| Field |
|---------|
| Payment ID |
| Order ID |
| Amount |
| Method |
| Status |

### Business Questions

- Successful payments
- Failed transactions
- Payment method analysis

---

## addresses.csv

Stores customer delivery addresses.

### Example Data

| Field |
|---------|
| Address ID |
| Customer ID |
| City |
| State |
| Country |

### Business Questions

- Geographic sales analysis
- Delivery region insights

---

## returns.csv

Stores information about returned orders.

### Example Data

| Field |
|---------|
| Return ID |
| Order ID |
| Return Date |
| Return Reason |
| Refund Amount |

### Business Questions

- Return rate analysis
- Product quality monitoring
- Refund tracking

---

## products.csv

Stores the complete product catalog.

### Example Data

| Field |
|---------|
| Product ID |
| Product Name |
| Category |
| Price |
| Supplier ID |

### Business Questions

- Product performance
- Category revenue
- Inventory analysis

---

## suppliers.csv

Stores supplier information.

### Example Data

| Field |
|---------|
| Supplier ID |
| Supplier Name |
| Country |
| Contact Information |

### Business Questions

- Supplier performance
- Supplier contribution to revenue

---

## payment_methods.csv

Lookup table containing payment method definitions.

### Examples

- Credit Card
- Debit Card
- UPI
- Cash on Delivery

### Purpose

Provides standardized payment categories.

---

## shipping_tier.csv

Lookup table containing shipping service levels.

### Examples

- Standard
- Express
- Same-Day

### Purpose

Provides standardized shipping classifications.

---

# Additional Data Sources Used Later

Although the first ten files come from PostgreSQL, we will also ingest data from the other three source systems.

| Data | Source | Bronze Location | Covered In |
|--------|--------|----------------|------------|
| Exchange Rates | Public API | raw/fx_rates/ | Week 1 Day 3 |
| Customer-Product Relationships | Neo4j | raw/graph/ | Week 1 Day 3 |
| Images and PDFs | Unstructured Files | raw/unstructured/ | Week 1 Day 3 |

---

# How the Tables Connect

The dataset follows a relational model where tables are connected through keys.

```text
customers
    |
    +------> orders
                |
                +------> order_items ------> products ------> suppliers
                |
                +------> payments
                |              |
                |              +------> payment_methods
                |
                +------> returns
                |
                +------> shipping_tier

customers
    |
    +------> addresses
```

---

# The Most Important Table: orders

The **orders** table sits at the center of the business model.

Most business questions start with orders and then connect to other tables.

Examples:

### Revenue Analysis

```text
orders
```

### Revenue by Product Category

```text
orders
   ↓
order_items
   ↓
products
```

### Revenue by Customer Location

```text
orders
   ↓
customers
   ↓
addresses
```

### Return Rate by Product

```text
orders
   ↓
returns
   ↓
order_items
   ↓
products
```

---

# Why We Need a Lakehouse

Consider the business question:

> What was last month's revenue by product category?

To answer this, we must combine:

```text
orders
   ↓
order_items
   ↓
products
```

This requires joining multiple tables before producing a result.

As datasets grow to millions or billions of records, these joins become expensive and slow.

The purpose of the Lakehouse is to:

- Centralize data
- Improve performance
- Simplify analytics
- Create trusted business-ready tables

---

## Key Takeaway

All 10 CSV files represent exports from PostgreSQL.

Together they form the core business dataset that we will:

1. Ingest into Bronze
2. Clean in Silver
3. Transform into Gold business models
4. Use for reporting and analytics

Everything we build throughout the bootcamp starts with these tables.

---
## Section 5: What We Build Over 3 Weeks

**INSTRUCTOR NOTE:**  
Show students the full picture of what they will build. This gives them motivation - by the end they will have a complete production-grade data platform.

---

```
+========================================================================+
|              GLOBALMART LAKEHOUSE - COMPLETE ARCHITECTURE              |
+========================================================================+
|
|  WEEK 1: Foundation + Ingestion
|  --------------------------------
|  - Set up ADLS Gen2 (Bronze / Silver / Gold)
|  - Connect Databricks to ADLS
|  - Write Delta tables (Medallion Architecture)
|  - Ingest from OLTP, CSV files, REST APIs
|  - Code versioning with GitHub
|  - Idempotent pipelines
|
|  WEEK 2: Data Quality + Transformation + Modelling
|  --------------------------------------------------
|  - CDC (Change Data Capture) - capture every row change
|  - Schema Evolution & Autoloader
|  - Data Quality checks and constraints
|  - Transformation patterns (Standardize, Conform, Enrich)
|  - Data Modelling (Kimball: Facts + Dimensions)
|  - SCD Type 1 and Type 2 using MERGE
|  - Incremental loading (Watermark + CDF)
|
|  WEEK 3: Orchestration + Governance + Optimization
|  --------------------------------------------------
|  - Orchestration with Airflow (automated daily pipelines)
|  - Error handling, DLQ, SLAs
|  - Data Governance (Unity Catalog, permissions, row-level security)
|  - Performance optimization (Z-order, Liquid Clustering, VACUUM)
|  - Genie + Semantic Layer for business users
|  - GraphFrames Analytics
|
+========================================================================+
|  END STATE: A fully automated, governed, production-grade Lakehouse    |
|  that ingests from 4 sources, cleans data, models it, and serves       |
|  business reports - running automatically every day.                   |
+========================================================================+
```

---

### Today's Focus (Day 1):

| Time | Session | What we set up |
|------|---------|---------------|
| 9:30 - 10:30 | ILT 1 (this session) | Understand the problem and the architecture |
| 11:00 - 12:00 | ILT 2 | Azure Cloud, ADLS Gen2, Lakehouse vs Warehouse |
| 12:00 - 1:00 | ILT 3 | Medallion Architecture and Delta Lake basics |
| 2:00 onwards | Hands-on | Create ADLS, upload data, connect Databricks, write first Delta tables |

**INSTRUCTOR NOTE:**  
Tell students: *'By the end of today, you will have your own ADLS Gen2 storage set up, all 10 GlobalMart files uploaded, and you will have written your first Delta table. That is a big achievement for Day 1.'

---
## Section 6: Live Demo - A Quick Look at the GlobalMart Data

**INSTRUCTOR NOTE:**  
Last 10 minutes of this session. Run these code cells to show students what the actual data looks like. This makes the abstract problem statement concrete.  
Keep it quick - we are just peeking at the data, not transforming anything yet.

Make sure your cluster is running before you start.

In [ ]:
# ============================================================
# CELL 1: Connect to ADLS and define paths
#
# INSTRUCTOR NOTE:
#   Same setup as ILT 2. Paste your storage key and run.
# ============================================================

storage_account_name = "amazonprojectadls"
container_name       = "amazon-data"
raw_folder           = "raw"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"   # paste your key here

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

raw_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{raw_folder}"

print("Connected to ADLS.")
print(f"Reading from: {raw_path}")

In [ ]:
# ============================================================
# CELL 2: Get a row count for all 10 tables
#
# INSTRUCTOR NOTE:
#   Show students the scale of data we are working with.
#   Notice products.csv has the most rows - it is the full product catalog.
#   orders_items has more rows than orders because each order has multiple products.
# ============================================================

tables = [
    "customers", "orders", "orders_items", "products",
    "payments", "addresses", "returns", "suppliers",
    "payment_methods", "shipping_tier"
]

print("GlobalMart Dataset Overview")
print("=" * 45)
print(f"{'Table':<25} {'Rows':>10}  Source")
print("-" * 55)

source_map = {
    "customers": "OLTP DB", "orders": "OLTP DB",
    "orders_items": "OLTP DB", "payments": "OLTP DB",
    "addresses": "OLTP DB", "returns": "OLTP DB",
    "products": "Flat File", "suppliers": "Flat File",
    "payment_methods": "Flat File", "shipping_tier": "Flat File"
}

for table in tables:
    df = spark.read.option("header", "true").csv(f"{raw_path}/{table}.csv")
    count = df.count()
    print(f"{table:<25} {count:>10,}  {source_map[table]}")

print("-" * 55)

In [ ]:
# ============================================================
# CELL 3: Peek at customers - who are they?
#
# INSTRUCTOR NOTE:
#   Show this to make the data human. These are the people
#   buying from GlobalMart. Point out columns - what is useful?
#   What might be messy? (nulls, email formats, etc.)
#   We will fix these issues in Week 2 (Silver transformation).
# ============================================================

df_customers = spark.read.option("header", "true").csv(f"{raw_path}/customers.csv")

print("Sample customers:")
df_customers.show(5, truncate=True)

print("Columns:", df_customers.columns)

In [ ]:
# ============================================================
# CELL 4: Peek at orders - what does a transaction look like?
#
# INSTRUCTOR NOTE:
#   Orders is the central table - most analysis will start here.
#   Notice the 'status' column - orders can be pending, shipped,
#   delivered, cancelled. This is a key field for business analytics.
# ============================================================

df_orders = spark.read.option("header", "true").csv(f"{raw_path}/orders.csv")

print("Sample orders:")
df_orders.show(5, truncate=True)

print("Columns:", df_orders.columns)

In [ ]:
# CELL 5: Show the business problem - data is in silos right now
#
# INSTRUCTOR NOTE:
#   This cell demonstrates WHY we need a Lakehouse.
#   The manager's question: 'How many orders does each customer have?'
#   Right now we CAN answer this because we have both tables.
#   But imagine if customers were in one database and orders in another
#   system with no connection - this join would be impossible without
#   a unified platform.
#
#   This is a preview of what Silver and Gold will look like.
# ============================================================

# A simple business question: which customers placed the most orders?
# ============================================================
# CELL 5: Show the business problem - data is in silos right now
# ============================================================

print("Business Question: Which customers placed the most orders?")
print("(This requires joining customers + orders - two separate source tables)")
print()

# Read source tables
df_customers = spark.read.option("header", "true").csv(f"{raw_path}/customers.csv")
df_orders = spark.read.option("header", "true").csv(f"{raw_path}/orders.csv")

from pyspark.sql.functions import count, col

# Count orders per customer
df_order_counts = (
    df_orders
    .groupBy("CustomerID")
    .agg(count("OrderID").alias("total_orders"))
)

# Join with customer information
df_result = (
    df_order_counts
    .join(df_customers, on="CustomerID", how="left")
    .select(
        "CustomerID",
        "FirstName",
        "LastName",
        "total_orders"
    )
    .orderBy(col("total_orders").desc())
)

print("Top 10 customers by order count:")

display(df_result.limit(10))

# print()
# print("This is what Gold layer will look like - pre-joined, aggregated, business-ready.")

# print()
# print("This is what Gold layer will look like - pre-joined, aggregated, business-ready.")

---
## Session Summary

| Topic | Key Takeaway |
|-------|-------------|
| **GlobalMart** | E-commerce company with data scattered across 4 separate systems |
| **The problem** | Data silos → no single truth → slow decisions |
| **Source 1: OLTP DB** | Live transactional database - orders, customers, payments |
| **Source 2: REST APIs** | External services returning JSON - reviews, shipping, promotions |
| **Source 3: Flat Files** | CSV exports from vendors - products, suppliers, config tables |
| **Source 4: Event Streams** | Real-time user actions - clicks, cart events, page views |
| **Central table** | `orders` - almost everything connects to an order |
| **The solution** | A Lakehouse on ADLS Gen2 that unifies all 4 sources |
| **3-week goal** | Fully automated, governed Lakehouse serving business dashboards |

---

### What Is Coming Next:

| Session | Topic |
|---------|-------|
| **ILT 2 (11:00 AM)** | Azure Cloud, ADLS Gen2, Lakehouse vs Warehouse vs Hybrid |
| **ILT 3 (12:00 PM)** | Medallion Architecture and Delta Lake Fundamentals |
| **Hands-On (2:00 PM)** | Create ADLS, upload data, connect Databricks, write first Delta table |

---

**INSTRUCTOR NOTE:**  
Closing activity (5 minutes): Ask each student to answer one question out loud:
1. *'Name one of the 4 data sources GlobalMart has.'*
2. *'What is one business problem that GlobalMart cannot solve right now?'*
3. *'What are the 3 layers of the Medallion Architecture called?'* (Bronze, Silver, Gold - even though we haven't covered it yet - some students may guess correctly)